# 数据质量与可复现流程

学习目标：把表格要求写成可检查的规则，保留错误来源，比较处理结果，并区分流程可复现与特定步骤的幂等性。

前置知识：列类型、缺失处理、条件筛选、连接、分组、Python 函数和异常处理。

运行环境：Python 3.12、pandas 3.0；示例按 pandas 3.0.6 编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制订单。列名、类型、数量范围和保留策略是本章明确制定的业务约定；pandas 提供实施与检查这些规则的工具。后续单元沿用 pd。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 找出不符合数量要求的记录

先明确一项规则：每条订单必须有数量，且数量在 1 至 100 件之间。notna 检查必填，between 检查闭区间。可空数值比较可能产生未知结果，本例将未知明确视为未通过。

错误表保留原输入的行标签，便于找到问题来源；不要先删除异常再声称输入全部合格。

In [1]:
import pandas as pd

orders = pd.DataFrame({"order_id": ["O1", "O2", "O3", "O4"],
                       "quantity": pd.array([2, None, 101, 4], dtype="Int64")},
                      index=["source-1", "source-2", "source-3", "source-4"])
valid_quantity = orders["quantity"].notna() & orders["quantity"].between(1, 100).fillna(False)
errors = orders.loc[~valid_quantity]
print(errors)  # source-2 缺失，source-3 的 101 超出范围。
print(valid_quantity.tolist())  # [True, False, False, True]。
print(len(orders), len(errors))  # 原始 4 行、错误 2 行，原表没有被删改。

         order_id  quantity
source-2       O2      <NA>
source-3       O3       101
[True, False, False, True]
4 2


## 2 先检查结构，再检查值

### 2.1 列名与类型

缺少列、同名列重复或类型不符时，后面的逐行规则可能无法正确解释数据。下面约定列顺序为 order_id、quantity、region，编号和地区使用 pandas str，数量使用可空 Int64。

列顺序属于本例输出约定，不是所有数据处理任务都必须固定。is_dtype_equal 比较实际 dtype 与目标类型，而不是从显示值猜测。

In [2]:
valid = pd.DataFrame({"order_id": ["O1", "O2"],
                      "quantity": pd.array([2, 3], dtype="Int64"), "region": ["east", "west"]},
                     index=["R1", "R2"])
expected_columns = ["order_id", "quantity", "region"]
expected_types = {"order_id": "str", "quantity": "Int64", "region": "str"}
print(valid.columns.is_unique, valid.columns.tolist() == expected_columns)  # True True。
type_checks = {name: pd.api.types.is_dtype_equal(valid[name].dtype, dtype)
               for name, dtype in expected_types.items()}
print(type_checks)  # 三列均为 True。
print(valid.dtypes)  # str、Int64、str；外观都是整数也不能代替类型检查。

True True
{'order_id': True, 'quantity': True, 'region': True}
order_id      str
quantity    Int64
region        str
dtype: object


### 2.2 对结构错误明确报错

将短小的结构检查封装为函数，便于后续复用。这里使用显式 if 与 ValueError，错误会指向具体规则；assert 更适合本章的结果自查，Python 优化运行可能移除 assert，不宜把它作为必须执行的输入校验。

函数先确认列完整且唯一，再访问各列。下面沿用上节的结构约定。

In [3]:
def check_structure(frame):
    if not frame.columns.is_unique:
        raise ValueError("列名必须唯一")
    if frame.columns.tolist() != expected_columns:
        raise ValueError("列名或列顺序不符合约定")
    for name, dtype in expected_types.items():
        if not pd.api.types.is_dtype_equal(frame[name].dtype, dtype):
            raise ValueError(f"{name} 类型应为 {dtype}")
    return frame


print(check_structure(valid).shape)  # (2, 3)，合格表正常通过。
wrong_type = valid.astype({"quantity": "float64"})

# 预期 ValueError：quantity 类型应为 Int64。
check_structure(wrong_type)

(2, 3)


ValueError: quantity 类型应为 Int64

## 3 必填、键唯一与集合成员

结构合格后，再检查每条记录的业务含义。订单编号必须非缺失且非空白，并且唯一；地区只能属于 east、west；数量仍要求 1 至 100 件。

duplicated(subset=..., keep=False) 标出同一个键的所有重复记录，便于一起核查。它检查数据列，不等于检查 DataFrame 的行索引唯一性。下面行标签表示来源位置，与业务订单编号分开保存。

In [4]:
raw = pd.DataFrame({"order_id": ["O1", "O1", "O3", None, " "],
                    "quantity": pd.array([2, 3, None, 5, 101], dtype="Int64"),
                    "region": ["east", "west", "east", "north", None]},
                   index=["L1", "L2", "L3", "L4", "L5"])
check_structure(raw)
issues = pd.DataFrame({
    "missing_id": raw["order_id"].isna() | raw["order_id"].str.strip().eq(""),
    "duplicate_id": raw.duplicated("order_id", keep=False),
    "invalid_quantity": ~raw["quantity"].between(1, 100).fillna(False),
    "invalid_region": ~raw["region"].isin(["east", "west"]),
}, index=raw.index)
print(issues)  # L1/L2 重复；L3 数量缺失；L4 缺编号且地区非法；L5 空白编号、数量超限、地区缺失。
print(issues.any(axis=1).tolist())  # 五行都至少违反一条规则。
print(raw.index.is_unique)  # True；来源行唯一不代表业务编号唯一。

    missing_id  duplicate_id  invalid_quantity  invalid_region
L1       False          True             False           False
L2       False          True             False           False
L3       False         False              True           False
L4        True         False             False            True
L5        True         False              True            True
[True, True, True, True, True]
True


## 4 错误记录追溯与行数约束

把错误标记与原输入并排保存，能同时看到违规原因和原始值。一个记录可能违反多项规则，因此“错误条目数”和“错误记录数”不能混为一谈。

本例仅把记录分为通过与未通过两组，不丢行也不展开。两组行数之和应等于原输入行数，且没有同一来源记录被同时放入两组。

In [5]:
bad_mask = issues.any(axis=1)
error_records = raw.loc[bad_mask].join(issues.loc[bad_mask])
accepted = raw.loc[~bad_mask]
print(error_records[["order_id", "missing_id", "duplicate_id", "invalid_quantity", "invalid_region"]])
# 各行仍以 L1-L5 对应原输入；一个来源可以同时有多个 True。
print(len(error_records), int(issues.to_numpy().sum()))  # 5 条记录、8 个违规标记。
print(len(accepted) + len(error_records), len(raw))  # 5 5。
assert len(accepted) + len(error_records) == len(raw)
assert accepted.index.intersection(error_records.index).empty

   order_id  missing_id  duplicate_id  invalid_quantity  invalid_region
L1       O1       False          True             False           False
L2       O1       False          True             False           False
L3       O3       False         False              True           False
L4      NaN        True         False             False            True
L5                 True         False              True            True
5 8
5 5


## 5 连接后的关系检查

连接要求不仅是结果能生成，还包括键关系、未匹配项与预期行数。many_to_one 检查右表键唯一；indicator 区分已经匹配和未匹配；左连接右表唯一时，每个左记录应对应一行。

下面沿用有效表 valid，补充地区名称。缺少 west 字典项时，行数仍正确，却不能声称所有地区已成功解释。

In [6]:
regions = pd.DataFrame({"region": ["east"], "region_name": ["东区"]})
enriched = valid.merge(regions, on="region", how="left", validate="many_to_one", indicator=True)
print(enriched)  # O1 为 both；O2 的 west 为 left_only，地区名称缺失。
print(len(enriched) == len(valid))  # True，行数通过不等于匹配完整。
print(enriched.loc[enriched["_merge"] != "both", ["order_id", "region"]])  # O2、west。
assert len(enriched) == len(valid)

  order_id  quantity region region_name     _merge
0       O1         2   east          东区       both
1       O2         3   west         NaN  left_only
True
  order_id region
1       O2   west


## 6 比较结果的三个工具

### 6.1 equals 返回整体判断

equals 比较形状、标签和数据，对相同位置的缺失视为相等；对应数据列的 dtype 不同会使结果不等。它不提供详细差异，也不是任意元数据的完整检查。

下面比较值看起来一样但整数类型不同的两张表，再观察相同缺失位置。

In [7]:
integers = pd.DataFrame({"value": [1, 2]}, index=["A", "B"])
floats = integers.astype("float64")
print(integers.equals(floats))  # False，int64 与 float64 不同。
with_missing = pd.DataFrame({"value": [1.0, None]}, index=["A", "B"])
print(with_missing.equals(with_missing.copy()))  # True，相同缺失位置视为一致。
print(integers.equals(integers.iloc[::-1]))  # False，记录顺序不同。

False
True
False


### 6.2 compare 展示数值差异

compare 要求形状和标签顺序一致，默认只展示有变化的行列，self 与 other 分别表示两份输入。双方同位置都是缺失时不列为差异。

下面仍使用 with_missing，修改一个有效值；compare 用于定位值差异，类型规则另行检查。

In [8]:
changed = with_missing.copy()
changed.loc["A", "value"] = 1.5
print(with_missing.compare(changed, result_names=("before", "after")))
# 只显示 A 的 value 从 1.0 变为 1.5，B 的共同缺失不列出。

   value      
  before after
A    1.0   1.5


### 6.3 assert_frame_equal 给出检查失败

assert_frame_equal 可以同时检查 dtype、标签、轴名称及数值，不满足条件时抛 AssertionError。严格比较时显式指定 check_exact=True；浮点近似比较应按任务允许误差给出 rtol、atol。

rtol 是相对容差，atol 是绝对容差。这里以右表数值作为参考，允许差值不超过“atol + rtol × 参考值的绝对值”，不把放宽误差当成修复错误。

In [9]:
reference = pd.DataFrame({"value": [0.3, 1.0]}, index=["A", "B"])
computed = pd.DataFrame({"value": [0.1 + 0.2, 1.0]}, index=["A", "B"])

# 预期 AssertionError：0.1+0.2 与 0.3 的二进制浮点值不完全相同，严格数值比较失败。
pd.testing.assert_frame_equal(computed, reference, check_exact=True)

AssertionError: DataFrame.iloc[:, 0] (column name="value") are different

DataFrame.iloc[:, 0] (column name="value") values are different (50.0 %)
[index]: [A, B]
[left]:  [0.30000000000000004, 1.0]
[right]: [0.3, 1.0]

In [10]:
pd.testing.assert_frame_equal(computed, reference, check_exact=False, rtol=0, atol=1e-12)
print("在本例明确的绝对误差范围内一致")  # 预期：数值差在本例容差内，断言通过后显示此提示。

在本例明确的绝对误差范围内一致


## 7 顺序约定与标签对应

若输出约定要求固定顺序，就应检查该顺序。若只关心同一个唯一键对应的数据，可以先确认键集合一致，再按键排序比较，或在适合的场景使用 check_like=True 忽略轴顺序。

忽略顺序仍要求标签与值对应，不能把每列各自排序后声称两表相同。下面用唯一订单编号作为索引进行比较。

In [11]:
before = valid.set_index("order_id")
after = before.iloc[::-1]
print(before.equals(after))  # False，顺序不同。
pd.testing.assert_frame_equal(before, after, check_like=True, check_exact=True)
print("忽略轴顺序后，每个订单仍对应相同数据")  # 预期：按标签对齐比较通过后显示此提示。
assert before.index.is_unique and after.index.is_unique
assert set(before.index) == set(after.index)
pd.testing.assert_frame_equal(before.sort_index(), after.sort_index(), check_exact=True)
print("明确按订单编号排序后也一致")  # 预期：两表显式排序后的比较通过后显示此提示。

False
忽略轴顺序后，每个订单仍对应相同数据
明确按订单编号排序后也一致


## 8 用短函数组织流程

pipe 把整个 DataFrame 交给函数，并传入额外参数；它不等于逐行 apply，也不自动做数据质量检查。先写清每一步，再用短链连接已有函数。

下面只处理列齐全、类型已经确定的输入。规范化函数返回新表，保留原始数据；区域仅去首尾空格并转大写。随后按显式参数计算运费，金额单位为分。

In [12]:
def normalize_regions(frame):
    result = frame.copy()
    result["region"] = result["region"].str.strip().str.upper()
    return result


def add_shipping(frame, cents_per_item):
    return frame.assign(shipping_cents=frame["quantity"] * cents_per_item)


input_table = pd.DataFrame({"order_id": ["O1", "O2"], "quantity": pd.array([2, 3], dtype="Int64"),
                            "region": [" east ", "West"]}, index=["R1", "R2"])
snapshot = input_table.copy()
normalized = normalize_regions(input_table)
result = add_shipping(normalized, cents_per_item=25)
via_pipe = input_table.pipe(normalize_regions).pipe(add_shipping, cents_per_item=25)
print(via_pipe)  # region 为 EAST、WEST；运费 50、75 分，原行标签保留。
pd.testing.assert_frame_equal(result, via_pipe, check_exact=True)
pd.testing.assert_frame_equal(input_table, snapshot, check_exact=True)
print("两种组织方式结果相同，原始输入未改变")  # 预期：管道与分步结果相同，输入未被修改，断言通过后显示此提示。

   order_id  quantity region  shipping_cents
R1       O1         2   EAST              50
R2       O2         3   WEST              75
两种组织方式结果相同，原始输入未改变


## 9 从同一输入复现

本章将可复现性约定为：从相同原始输入出发，在约定环境和相同参数下重新运行，得到满足比较规则的一致结果。应保存原输入，明确排序、缺失、连接、分组等参数；有随机抽样时还需固定输入顺序与 random_state。

它不要求把结果再送回流程后不变。下面对同一个 input_table 运行两次短流程，并检查原输入和输出结构。

In [13]:
first_run = input_table.pipe(check_structure).pipe(normalize_regions).pipe(add_shipping, cents_per_item=25)
second_run = input_table.pipe(check_structure).pipe(normalize_regions).pipe(add_shipping, cents_per_item=25)
pd.testing.assert_frame_equal(first_run, second_run, check_exact=True)
pd.testing.assert_frame_equal(input_table, snapshot, check_exact=True)
assert len(first_run) == len(input_table)
assert first_run["region"].isin(["EAST", "WEST"]).all()
print(first_run["shipping_cents"].tolist(), first_run.dtypes.tolist())  # 50、75；运费仍为 Int64。
print("同一输入与参数得到一致结果")

[50, 75] [<StringDtype(na_value=nan)>, Int64Dtype(), <StringDtype(na_value=nan)>, Int64Dtype()]
同一输入与参数得到一致结果


## 10 只对约定步骤检查幂等性

某步骤对自己的结果再次执行仍不改变结果，称为幂等。本例的地区规范化应满足这一要求；它不是全部数据处理操作共有的保证。

先明确哪些步骤需要幂等，再检查这些步骤。下面继续使用 normalize_regions；两次都只统一空白和大小写。

In [14]:
once = normalize_regions(input_table)
twice = normalize_regions(once)
pd.testing.assert_frame_equal(once, twice, check_exact=True)
print(once["region"].tolist(), twice["region"].tolist())  # 均为 EAST、WEST。
print("地区规范化步骤满足本例的幂等要求")

['EAST', 'WEST'] ['EAST', 'WEST']
地区规范化步骤满足本例的幂等要求


### 10.1 去重需要明确保留规则

drop_duplicates 默认比较数据列，不使用行索引；subset 可以限定键，keep 决定保留首条、末条或全部删除重复项。规则必须先决定，不能仅因为去重后没有重复就认定保留正确。

下面明确约定：相同订单的记录按输入顺序最后一条生效。保留来源标签，对这个去重步骤单独检查再次执行不变。

In [15]:
revisions = pd.DataFrame({"order_id": ["O1", "O2", "O1"], "quantity": [2, 3, 5]},
                          index=["L1", "L2", "L3"])
latest = revisions.drop_duplicates("order_id", keep="last")
again = latest.drop_duplicates("order_id", keep="last")
print(latest)  # 保留 L2 的 O2=3 和 L3 的 O1=5；本例依据输入顺序，不是自动识别更新时间。
pd.testing.assert_frame_equal(latest, again, check_exact=True)
print(latest["order_id"].is_unique)  # True；仍需另行确认“最后一条生效”确实是业务规则。

   order_id  quantity
L2       O2         3
L3       O1         5
True


### 10.2 单位换算并不幂等

将摄氏度转为华氏度是输入单位明确的变换，不能把已转换的数值再当作摄氏度重复处理。下面只用一个数值说明：从同一原始输入反复计算可以一致，但把结果继续变换会不同。

因此，可复现检查与幂等检查必须使用不同的输入关系。

In [16]:
celsius = pd.Series([0.0, 10.0], name="temperature")
first = celsius * 9 / 5 + 32
repeat_from_raw = celsius * 9 / 5 + 32
wrong_second_conversion = first * 9 / 5 + 32
print(first.tolist(), repeat_from_raw.tolist())  # 都是 32、50 °F。
print(first.equals(repeat_from_raw))  # True，同一输入的重复计算一致。
print(wrong_second_conversion.tolist())  # 89.6、122；错误地把华氏数值当成摄氏输入。
print(first.equals(wrong_second_conversion))  # False，这个变换不满足幂等性。

[32.0, 50.0] [32.0, 50.0]
True
[89.6, 122.0]
False


## 本章小结

（1）先检查列结构与类型，再检查必填、唯一键、范围和集合成员；每项规则都要有明确业务含义。

（2）错误表保留来源，检查记录数守恒、连接关系和未匹配项。某一项通过不能替代其他质量要求。

（3）equals 用于整体判断，compare 展示值差异，assert_frame_equal 执行明确的比较规则；浮点容差和排序要求需单独约定。

（4）pipe 组织整表函数。可复现从同一原输入开始，幂等则把结果再次交给同一步骤，两者不能互相替代。

## 练习

（1）给下表生成逐行错误标记，并输出原始值与原因。编号必须唯一且非缺失；数量必须为 1 至 10；地区仅允许 east、west。要求保留原行标签，不先删除错误记录。

In [17]:
practice = pd.DataFrame({"order_id": ["A", "A", "B"], "quantity": pd.array([2, None, 11], dtype="Int64"),
                         "region": ["east", "west", "north"]}, index=["X1", "X2", "X3"])
# 在此检查类型和业务规则；X1/X2 键重复，X2 数量缺失，X3 数量及地区违规。
# 检查错误记录数与错误标记数不同，并说明定位每个错误所需的来源字段。

（2）先预测 equals 的两个结果，再运行。如果任务改为“同一标签对应值相同即可，允许行顺序变化”，应选择什么比较方式？解释为何不能随意忽略 dtype。

In [18]:
a = pd.DataFrame({"count": [1, 2]}, index=["A", "B"])
b = a.iloc[::-1]
c = a.astype("float64")
print(a.equals(b))
print(a.equals(c))
# 先记录预测；按新顺序约定完成一次 assert_frame_equal，并保留类型检查。

False
False


（3）写一个返回新表的函数，将地区去空白并转大写，用 pipe 调用。分别检查从相同原始输入运行两次，以及把第一次输出再次规范化；解释这两个检查各证明什么。

In [19]:
source = pd.DataFrame({"region": [" east ", "West", None]})
# 在此实现并调用短函数；检查原表未变，缺失保留，两个目标分别得到相同结果。
# 用注释分别命名可复现与幂等，不将这项结论推广到所有 pipe 流程。

（4）原任务规定同一订单第一条生效，后来改成最后一条生效。修改去重参数，分别核对保留来源与再次去重的结果，解释为什么“两种结果都幂等”不能证明二者都符合当前业务要求。

In [20]:
versions = pd.DataFrame({"order_id": ["O1", "O1"], "quantity": [2, 5]}, index=["old", "new"])
# 在此对照 keep='first' 与 keep='last'。
# 检查：分别保留 old=2 和 new=5；当前规则要求后者，还需保留原始输入用于追溯。

### 重点练习提示（第 1 题）

提示一：把每条规则做成同索引的布尔列，让一条记录能同时带多个原因。

提示二：重复键使用 duplicated(..., keep=False) 标记全部成员；数量缺失与越界分开统计。

### 参考解析（第 1 题）

建立五项标记：编号缺失、编号重复、数量缺失、非缺失数量不在 1 至 10、地区不在 east/west。对本题输入，X1 有编号重复；X2 有编号重复和数量缺失；X3 有数量越界和地区非法。错误记录共 3 条，错误标记共 5 个。数量越界用 notna 与取反后的 between 组合，并明确把未知布尔结果填为 False，避免把同一个缺失再算作越界。按这些标记的逐行 any 选择问题记录，同时保留 X1 至 X3、原值和每项原因；先删掉重复行会丢失定位信息。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档或 API 源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [is_dtype_equal](https://pandas.pydata.org/docs/reference/api/pandas.api.types.is_dtype_equal.html) 的类型比较；[Series.between](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html) 的边界与缺失，结合显式填充规则；[Series.isin](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) 的集合成员；[DataFrame.duplicated](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) 与 [drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) 的 subset、keep、索引不参与；[DataFrame.equals](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.equals.html) 的形状、dtype、标签与缺失；[DataFrame.compare](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.compare.html) 的相同标签要求与共同缺失；[assert_frame_equal](https://pandas.pydata.org/docs/reference/api/pandas.testing.assert_frame_equal.html) 的 check_dtype、check_names、check_exact、check_like、rtol、atol；[DataFrame.pipe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pipe.html) 的整表函数输入与参数传递；[copy](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.copy.html)、[str.strip](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html)、[str.upper](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.upper.html) 的副本与文本处理；[merge](https://pandas.pydata.org/docs/reference/api/pandas.merge.html) 的 validate、indicator 与 left 连接。质量规则、可复现范围及幂等要求是本章应用约定，不是全部 pandas 操作共有的保证。 |
| Python 官方文档（3.12） | [The assert statement](https://docs.python.org/3.12/reference/simple_stmts.html#the-assert-statement)：优化模式可能不生成 assert 代码，用于区分结果自查与必要输入检查。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 API 源码与 docstring：[pandas.api.types.is_dtype_equal](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/dtypes/common.py#L663-L725)、[pandas.Series.between](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/series.py#L6081-L6170)、[pandas.Series.isin](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/series.py#L6002-L6079)、[pandas.DataFrame.duplicated](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/frame.py#L7965-L8097)、[pandas.DataFrame.drop_duplicates](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/frame.py#L7861-L7963)、[pandas.DataFrame.equals](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/generic.py#L1356-L1451)、[pandas.DataFrame.compare](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/frame.py#L10028-L10178)、[pandas.testing.assert_frame_equal](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/_testing/asserters.py#L1148-L1362)、[pandas.DataFrame.pipe](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/generic.py#L6012-L6114)、[pandas.DataFrame.copy](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/generic.py#L6548-L6654)、[pandas.Series.str.strip](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/strings/accessor.py#L2475-L2553)、[pandas.Series.str.upper](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/strings/accessor.py#L3924-L3994)、[pandas.merge](https://github.com/pandas-dev/pandas/blob/v3.0.6/pandas/core/reshape/merge.py#L145-L399)；对应上列 API 的参数和行为说明。 |